### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

---

# 1.3: Evaluation Refinement (phase 1)

---



## 💬 Product Team Check-in #1
*You share the initial eval results with your product team. The Correctness scores are reasonable, but the team has been collecting user feedback and has some specific findings.*

---

> **Sam (Product):** "Hey — the beta testers in ML Engineering have been active. Two pieces of feedback worth acting on. First, the tone feels a bit robotic. Users said it feels like they're reading documentation rather than talking to a helpful assistant."
>
> **You:** "Easy fix — that's a system prompt change plus a `Guidelines` scorer to catch it. I'll update the prompt to be more conversational and add an eval case or two that would fail with the current tone."
>
> **Sam:** "Great. Second thing — users are asking non-MLflow questions and the agent is just... answering them. Someone asked for a pasta recipe and it gave them one."
>
> **You:** "That's another possible system prompt guardrail. I'll add an explicit instruction to stay on topic and add a `Guidelines` scorer for it. I'll also add an eval case with an off-topic question so we can catch regressions if the guardrail ever slips."

---

Let's translate that into some evaluations.

## 1 - Environment Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3-flash-preview"

mlflow_tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(mlflow_tracking_uri)

mlflow.set_experiment("mlflow-agent-1")
mlflow.openai.autolog()

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:5000)")

Introduce Guidlines Scorer

# Problem Validation

In [ ]:
eval_dataset = [
    # Baseline
    # Off-topic — bare LLM will just answer this
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {
            "guidelines": [
                "Response must not provide a pasta recipe or any cooking-related content",
            ],
        },
    },
    {
        "inputs": {"question": "Is there a seahorse emoji?"},
        "expectations": {
            "guidelines": [
                "Response must not answer questions that aren't related to MLflow",
            ],
        },
    },
]


In [ ]:
# Evaluate

# Prompt Registration

---
## The Problem: Prompts as Strings

Our system prompt in its current form:

```python
SYSTEM_PROMPT = """You are a helpful MLflow assistant..."""

# After some feedback
SYSTEM_PROMPT_V2 = """You are a helpful MLflow assistant...Always include code examples and version context."""
```

Questions this process can't answer:
- Which prompt version was running when User #347 got a wrong API recommendation?
- Did changing "be practical" to "include code examples" improve Correctness scores?
- Can we roll back to last week's prompt without a code deploy?

The prompt registry makes all of this much more streamlined.

In [ ]:
SYSTEM_PROMPT = """You are a helpful MLflow assistant."""

---
## 2 — Registering the System Prompt

`mlflow.genai.register_prompt()` creates a new prompt (or a new version of an existing one) in the registry. The prompt is stored in the tracking server's database and is immediately available to any notebook, script, or service that can reach the server.

Let's register the original system prompt from the first notebook — the bare-bones version before any of Sam's feedback.

In [ ]:
prompt_v1 = mlflow.genai.register_prompt(
    name="mlflow-agent-system",
    template=(SYSTEM_PROMPT),
    commit_message="Initial system prompt — bare PoC version",
    tags={
        "author": "ml-team",
        "agent": "mlflow-agent",
        "stage": "dev",
    },
)

print(f"Prompt: {prompt_v1.name}")
print(f"Version: {prompt_v1.version}")
print(f"Template: {prompt_v1.template[:80]}...")

In [ ]:
eval_dataset = [
    # Baseline example from Stage 1
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.log_metric", "key", "value"]
        },
    },
    # API specificity — requires a real function signature, not a vague description
    {
        "inputs": {"question": "How do I run an evaluation in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.genai.evaluate", "data", "scorers"],
            "guidelines": [
                "Response must include a concrete code example, not just a description of what to do",
                "Response must reference specific parameter names, not just say 'pass in your data'",
            ],
        },
    },
    # Version clarity — requires distinguishing 3.0 from pre-3.0
    {
        "inputs": {"question": "How do I add a custom scorer to my evaluation?"},
        "expectations": {
            "expected_facts": ["@scorer", "mlflow.genai"],
            "guidelines": [
                "Response must indicate this API is specific to MLflow 3.0",
                "Response must not reference mlflow.evaluate() which is the pre-3.0 pattern",
            ],
        },
    },
    # Tone — conversational vs. changelog
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
        "expectations": {
            "guidelines": [
                "Response should be conversational and helpful, not a dry enumeration of features",
                "Response should explain the practical benefit to the user, not just what the feature does",
            ],
        },
    },
    # Combines all three — specificity, version clarity, and tone
    {
        "inputs": {"question": "How do I trace a custom function in MLflow 3.0?"},
        "expectations": {
            "expected_facts": ["@mlflow.trace", "SpanType"],
            "guidelines": [
                "Response must include a code example showing the decorator in use",
                "Response must clarify this is a MLflow 3.0 feature",
                "Response should be conversational and explain when you would want to do this",
            ],
        },
    },
]
print(f"Eval set size: {len(eval_dataset)} examples")

In [ ]:
eval_dataset_v4 = [
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.log_metric", "key", "value", "step"],
        },
    },
    {
        "inputs": {"question": "How do I set up autologging for a scikit-learn model?"},
        "expectations": {
            "expected_response": "To enable autologging for scikit-learn in MLflow, call mlflow.sklearn.autolog() before your training code. This automatically logs parameters, metrics, and the trained model to your active run.",
        },
    },
    {
        "inputs": {"question": "How do I organize my runs in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.set_experiment", "experiment", "run"],
        },
    },
    {
        "inputs": {"question": "How do I log a model artifact in MLflow?"},
        "expectations": {
            "expected_response": "You can log a model artifact using mlflow.log_artifact() for individual files, or framework-specific methods like mlflow.sklearn.log_model() which saves the model along with its dependencies and a model signature.",
        },
    },
    {
        "inputs": {"question": "How do I compare runs in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.search_runs", "experiment_id", "metrics"],
        },
    },
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {
            "guidelines": [
                "Response must not provide a pasta recipe or any cooking-related content",
            ],
        },
    },
    {
        "inputs": {"question": "Is there a seahorse emoji?"},
        "expectations": {
            "guidelines": [
                "Response must not answer questions that aren't related to MLflow",
            ],
        },
    },
]
print(f"Eval set size: {len(eval_dataset_v4)} examples")

---

## 3 - Update System Prompt

In [ ]:
SYSTEM_PROMPT_V2 = """You are a helpful MLflow assistant.
Answer questions about MLflow APIs, tracking, tracing, evaluation, and model management.
Do not answer questions that aren't related to MLflow."""

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name="mlflow-agent-system",
    template=(SYSTEM_PROMPT_V2),
    commit_message="Added code examples, version context, and conversational tone per Sam's feedback",
    tags={
        "author": "ml-team",
        "agent": "mlflow-agent",
        "stage": "dev",
    },
)

print(f"Prompt: {prompt_v2.name}")
print(f"Version: {prompt_v2.version}")
print(f"Template: {prompt_v2.template[:80]}...")

In [ ]:
# Update the predict_fn to use the new system prompt
def mlflow_agent(question: str) -> str:
    """MLflow assistant with updated prompt."""
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_V2},
            {"role": "user",   "content": question},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

In [ ]:
# Evaluate

---
## 4 — Adding the Scorers Sam Asked For

Now we put the pre-built scorers from Section 3 to work. `Guidelines` is the workhorse — it's how we encode Sam's product requirements as evaluatable constraints.

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Guidelines

results_v2 = mlflow.genai.evaluate(
    data=eval_dataset_v4,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="conversational_tone",
            guidelines=(
                "Responses should feel like advice from a knowledgeable colleague, not a "
                "changelog or dry reference doc. Avoid overly formal language or bullet-point-only answers."
            ),
        ),
        Guidelines(
            name="stays_on_topic",
            guidelines=(
                "Responses must only address MLflow-related questions. If the user asks about "
                "something unrelated to MLflow, the response should politely decline and redirect "
                "the user to ask an MLflow-related question instead."
            ),
        ),
    ],
)
print("=== Run 2: Guidelines scorers added ===")
print(results_v2.metrics)

---
## 5 — Custom Scorers with `@scorer`

`Guidelines` scorers use an LLM judge, which makes them flexible but slow and non-deterministic. For rules that can be expressed as code, a `@scorer` function is faster, cheaper, and completely predictable.

Your function receives:
- `outputs` — the string returned by `predict_fn`
- `inputs` — the original input dict (keyword arg)
- `expectations` — the ground truth dict (keyword arg)

Return `bool`, `int`, `float`, or a `Feedback` object with a rationale.

In [ ]:
import re
from mlflow.genai.scorers import scorer, Feedback


@scorer
def has_code_block(outputs: str, **kwargs) -> bool:
    """
    Checks that the response includes at least one code-like snippet.
    Matches patterns like: 'mlflow.something(', 'import mlflow', '@mlflow.trace'.
    This is a fast, deterministic check — no LLM call needed.
    """
    code_pattern = re.compile(
        r'(mlflow\.\w+\(|import mlflow|@mlflow\.\w+|'
        r'from mlflow|\.log_\w+|\.evaluate\(|\.autolog\()',
        re.IGNORECASE
    )
    return bool(code_pattern.search(outputs))


@scorer
def not_overwhelming(outputs: str, **kwargs) -> Feedback:
    """
    Checks the response isn't too long. Users want focused answers, not essays.
    Returns a Feedback object so we can see the character count in the UI.
    """
    char_count = len(outputs)
    passes = char_count <= 2000
    return Feedback(
        value=passes,
        rationale=f"{char_count} characters — {'within' if passes else 'over'} the 2000-char limit",
    )


results_v4 = mlflow.genai.evaluate(
    data=eval_dataset_v2,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="includes_code_examples",
            guidelines=(
                "When explaining an MLflow API or feature, the response must include "
                "at least one concrete code snippet with real function names and parameter names."
            ),
        ),
        Guidelines(
            name="includes_version_context",
            guidelines=(
                "When referencing MLflow APIs, the response should mention which version "
                "of MLflow the API applies to or was introduced in."
            ),
        ),
        has_code_block,         # fast deterministic check
        not_overwhelming,       # fast deterministic check with rationale
    ],
)

print("=== Run 3: Custom scorers added ===")
print(results_v4.metrics)